# Notebook 1: Logistic Regression baseline (+ hypothesis-only LR)
**Kaggle: attach Dataset `nli-data`. No GPU needed (accelerator: None). Internet not needed.**

Protocol: features/vectorisers fitted on the 100K train set only. All choices (feature group, C) made on **SNLI validation only**.
The chosen model is evaluated once on SNLI test, MNLI-matched, MNLI-mismatched (never used for tuning).

Steps: (1) feature-group ablation on the 50K subset, (2) C grid on 100K with the winning feature group,
(3) final evaluation, (4) hypothesis-only LR (+ out-of-fold train probabilities for the novelty part).

In [1]:
import glob, json, os, re, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp, joblib
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

# Locate the shared module + data wherever Kaggle mounted the Dataset
hits = glob.glob(os.environ.get("NLI_ROOT", "/kaggle/input") + "/**/nli_common.py", recursive=True)
assert hits, "nli_common.py not found: attach the 'nli-data' Dataset to this notebook"
ROOT = Path(hits[0]).parent
sys.path.insert(0, str(ROOT))
DATA = str(ROOT / "data")
from nli_common import LABELS, SEED, load_split, save_run, evaluate
OUT = Path(os.environ.get("NLI_OUT", "/kaggle/working/results/lr")); OUT.mkdir(parents=True, exist_ok=True)
print("data dir:", DATA)

tr100, tr50 = load_split("train_100k", DATA), load_split("train_50k", DATA)
val, test = load_split("snli_val", DATA), load_split("snli_test", DATA)
mnli_m, mnli_mm = load_split("mnli_m", DATA), load_split("mnli_mm", DATA)
print({k: len(v) for k, v in dict(tr100=tr100, tr50=tr50, val=val, test=test, mnli_m=mnli_m, mnli_mm=mnli_mm).items()})

data dir: /kaggle/input/datasets/punyapratap07/nil-data/nli_data/data
{'tr100': 100000, 'tr50': 50000, 'val': 9842, 'test': 9824, 'mnli_m': 9815, 'mnli_mm': 9832}


## Features

In [2]:
TOKEN_PATTERN = r"[a-z0-9]+(?:'[a-z]+)?"
TOKEN_RE = re.compile(TOKEN_PATTERN)
NEG_WORDS = {"not", "no", "nobody", "nothing", "never", "none", "nowhere", "neither", "nor", "cannot"}
CROSS_MAX_NOVEL, CROSS_MAX_DROPPED = 10, 15
TFIDF_PARAMS = dict(max_features=50_000, ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=TOKEN_PATTERN)

def tokenize(t): return TOKEN_RE.findall(t.lower())
def has_neg(toks): return any(w in NEG_WORDS or w.endswith("n't") for w in toks)

def analyze(df):
    """novel words (in hypothesis, not premise), cross-pair tokens (dropped x novel), 14 dense features."""
    novel_t, cross_t, dense = [], [], []
    for prem, hyp in zip(df["premise"], df["hypothesis"]):
        p, h = tokenize(prem), tokenize(hyp)
        ps, hs = set(p), set(h)
        novel = [w for w in dict.fromkeys(h) if w not in ps]
        dropped = [w for w in dict.fromkeys(p) if w not in hs]
        inter, union = len(ps & hs), len(ps | hs)
        pb, hb = set(zip(p, p[1:])), set(zip(h, h[1:]))
        pn, hn = has_neg(p), has_neg(h)
        novel_t.append(" ".join(novel))
        cross_t.append(" ".join(f"{a}__{b}" for a in dropped[:CROSS_MAX_DROPPED] for b in novel[:CROSS_MAX_NOVEL]))
        dense.append([len(p), len(h), len(h) - len(p), min(len(h) / max(1, len(p)), 10.0),
                      inter / max(1, len(hs)), inter / max(1, len(ps)), inter / max(1, union),
                      len(novel), len(dropped), len(hb & pb) / max(1, len(hb)),
                      float(hn), float(pn), float(hn != pn), float(len(hs) > 0 and hs <= ps)])
    return novel_t, cross_t, np.asarray(dense, dtype=np.float64)

GROUPS = {   # cumulative feature groups for the ablation
    "concat":       ["P", "H"],
    "+diff_cos":    ["P", "H", "diff", "cos"],
    "+dense":       ["P", "H", "diff", "cos", "dense"],
    "+novel_cross": ["P", "H", "diff", "cos", "dense", "novel", "cross"],
}

class Feats:
    def fit(self, df):
        novel, _, dense = analyze(df)
        self.tfidf = TfidfVectorizer(**TFIDF_PARAMS).fit(pd.concat([df["premise"], df["hypothesis"]], ignore_index=True))
        self.novel = TfidfVectorizer(lowercase=False, token_pattern=r"\S+", min_df=2, sublinear_tf=True).fit(novel)
        self.scaler = StandardScaler().fit(dense)
        self.cross = HashingVectorizer(n_features=2 ** 19, binary=True, norm=None, alternate_sign=False,
                                       lowercase=False, token_pattern=r"\S+")
        return self

    def blocks(self, df):
        novel, cross, dense = analyze(df)
        P, H = self.tfidf.transform(df["premise"]), self.tfidf.transform(df["hypothesis"])
        return {"P": P, "H": H, "diff": abs(P - H),
                "cos": sp.csr_matrix(np.asarray(P.multiply(H).sum(axis=1))),   # rows are L2-normalised -> cosine
                "dense": sp.csr_matrix(self.scaler.transform(dense)),
                "novel": self.novel.transform(novel), "cross": self.cross.transform(cross).tocsr()}

def assemble(blocks, group): return sp.hstack([blocks[k] for k in GROUPS[group]], format="csr")

MAX_ITER = 2000   # only a cap: lbfgs stops as soon as tol is reached

def conv(clf):
    n = int(np.max(clf.n_iter_))
    return f"iters={n}" + ("" if n < MAX_ITER else " (HIT CAP, not converged)")

def fit_lr(X, y, C):
    t = time.time()
    clf = LogisticRegression(C=C, solver="lbfgs", max_iter=MAX_ITER, tol=1e-4, random_state=SEED).fit(X, y)
    return clf, time.time() - t

## 1. Feature-group ablation (50K subset, C=1, validation accuracy)

In [3]:
f50 = Feats().fit(tr50)
b50, bv50 = f50.blocks(tr50), f50.blocks(val)
abl = []
for g in GROUPS:
    Xtr, Xv = assemble(b50, g), assemble(bv50, g)
    clf, tt = fit_lr(Xtr, tr50["label"], C=1.0)
    m = evaluate(val["label"], clf.predict(Xv))
    abl.append({"group": g, "n_features": Xtr.shape[1], "val_acc": m["accuracy"], "val_f1_macro": m["f1_macro"], "fit_s": round(tt)})
    print(f"{g:<14} features={Xtr.shape[1]:>8,}  val acc={m['accuracy']:.4f}  macroF1={m['f1_macro']:.4f}  fit={tt:.0f}s  {conv(clf)}")
abl_df = pd.DataFrame(abl); abl_df.to_csv(OUT / "lr_feature_ablation.csv", index=False)
BEST_GROUP = abl_df.sort_values("val_acc", ascending=False).iloc[0]["group"]
print("\nSelected feature group (by validation accuracy):", BEST_GROUP)

concat         features= 100,000  val acc=0.6252  macroF1=0.6251  fit=17s  iters=141
+diff_cos      features= 150,001  val acc=0.6734  macroF1=0.6730  fit=22s  iters=133
+dense         features= 150,015  val acc=0.6955  macroF1=0.6951  fit=71s  iters=446
+novel_cross   features= 679,826  val acc=0.7306  macroF1=0.7303  fit=104s  iters=263

Selected feature group (by validation accuracy): +novel_cross


## 2. C grid on the full 100K train set (selected feature group)

In [4]:
f100 = Feats().fit(tr100)
b100, bval = f100.blocks(tr100), f100.blocks(val)
Xtr, Xval = assemble(b100, BEST_GROUP), assemble(bval, BEST_GROUP)
grid, fitted = [], {}
for C in [0.1, 1.0, 10.0]:
    clf, tt = fit_lr(Xtr, tr100["label"], C)
    m = evaluate(val["label"], clf.predict(Xval))
    fitted[C] = (clf, tt)
    grid.append({"C": C, "val_acc": m["accuracy"], "val_f1_macro": m["f1_macro"], "fit_s": round(tt)})
    print(f"C={C:<5} val acc={m['accuracy']:.4f}  macroF1={m['f1_macro']:.4f}  fit={tt:.0f}s  {conv(clf)}")
grid_df = pd.DataFrame(grid); grid_df.to_csv(OUT / "lr_C_grid.csv", index=False)
BEST_C = float(grid_df.sort_values("val_acc", ascending=False).iloc[0]["C"])
clf, train_time = fitted[BEST_C]
print("\nSelected C (by validation accuracy):", BEST_C)

C=0.1   val acc=0.7468  macroF1=0.7465  fit=104s  iters=208
C=1.0   val acc=0.7421  macroF1=0.7417  fit=181s  iters=353
C=10.0  val acc=0.7354  macroF1=0.7350  fit=191s  iters=377

Selected C (by validation accuracy): 0.1


## 3. Final evaluation (test sets touched once)

In [5]:
config = {"model": "LogisticRegression", "train_file": "snli_train_100k", "n_train": len(tr100), "seed": SEED,
          "feature_group": BEST_GROUP, "C": BEST_C, "solver": "lbfgs", "max_iter": MAX_ITER, "n_iter": int(np.max(clf.n_iter_)), "tol": 1e-4,
          "tfidf": {k: str(v) for k, v in TFIDF_PARAMS.items()}, "ablation_on": "snli_train_50k, C=1",
          "n_features": int(Xtr.shape[1])}
n_params = int(clf.coef_.size + clf.intercept_.size)
for name, df in [("snli_val", val), ("snli_test", test), ("mnli_m", mnli_m), ("mnli_mm", mnli_mm)]:
    t = time.time()
    X = assemble(f100.blocks(df), BEST_GROUP)
    probs = clf.predict_proba(X)
    save_run(OUT, "lr", name, df, probs, config, {"train_time_s": train_time, "inference_s": time.time() - t, "n_params": n_params})
joblib.dump({"feats": f100, "clf": clf, "group": BEST_GROUP}, OUT / "lr_model.joblib", compress=3)

[lr | snli_val] acc=0.7468 macroF1=0.7465 per-class F1=[0.791, 0.7, 0.749]
[lr | snli_test] acc=0.7456 macroF1=0.7454 per-class F1=[0.788, 0.697, 0.752]
[lr | mnli_m] acc=0.4918 macroF1=0.4889 per-class F1=[0.448, 0.5, 0.519]
[lr | mnli_mm] acc=0.4909 macroF1=0.4858 per-class F1=[0.412, 0.493, 0.553]


['/kaggle/working/results/lr/lr_model.joblib']

## 4. Hypothesis-only LR (annotation-artifact probe, used later for the hard/easy split and debiasing)

In [6]:
def hyp_pipe(C):
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=TOKEN_PATTERN),
                         LogisticRegression(C=C, solver="lbfgs", max_iter=MAX_ITER, random_state=SEED))

hres = []
for C in [0.1, 1.0, 10.0]:
    t = time.time(); pipe = hyp_pipe(C).fit(tr100["hypothesis"], tr100["label"])
    acc = evaluate(val["label"], pipe.predict(val["hypothesis"]))["accuracy"]
    hres.append((C, acc, pipe, time.time() - t)); print(f"hyp-only C={C:<5} val acc={acc:.4f}  {conv(pipe[-1])}")
H_C, h_acc, hpipe, h_time = max(hres, key=lambda r: r[1])
hconfig = {"model": "LogisticRegression (hypothesis-only)", "train_file": "snli_train_100k", "C": H_C, "seed": SEED,
           "features": "TF-IDF 1-2gram on hypothesis only"}
for name, df in [("snli_val", val), ("snli_test", test), ("mnli_m", mnli_m), ("mnli_mm", mnli_mm)]:
    save_run(OUT, "lr_hyponly", name, df, hpipe.predict_proba(df["hypothesis"]), hconfig, {"train_time_s": h_time})

# Out-of-fold probabilities on TRAIN (5-fold): an in-sample model would be over-confident on its own training rows.
oof = cross_val_predict(hyp_pipe(H_C), tr100["hypothesis"], tr100["label"],
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")
oof_df = tr100[["id", "label"]].copy(); oof_df[["p_ent", "p_neu", "p_con"]] = oof
oof_df.to_csv(OUT / "hyponly_train_oof.csv", index=False)
print("hyp-only OOF train acc:", round(float((oof.argmax(1) == tr100["label"].to_numpy()).mean()), 4))

hyp-only C=0.1   val acc=0.6240  iters=57
hyp-only C=1.0   val acc=0.6451  iters=166
hyp-only C=10.0  val acc=0.6263  iters=408
[lr_hyponly | snli_val] acc=0.6451 macroF1=0.6450 per-class F1=[0.656, 0.644, 0.635]
[lr_hyponly | snli_test] acc=0.6434 macroF1=0.6431 per-class F1=[0.654, 0.648, 0.628]
[lr_hyponly | mnli_m] acc=0.4082 macroF1=0.3642 per-class F1=[0.145, 0.49, 0.457]
[lr_hyponly | mnli_mm] acc=0.4155 macroF1=0.3674 per-class F1=[0.13, 0.495, 0.477]
hyp-only OOF train acc: 0.6277


## 5. Why does LR drop on MNLI? Vocabulary coverage

In [7]:
vocab = {w for w in f100.tfidf.vocabulary_ if " " not in w}
def coverage(df):
    toks = [t for s in pd.concat([df["premise"], df["hypothesis"]]) for t in tokenize(s)]
    return float(np.mean([t in vocab for t in toks]))
cov = {n: round(coverage(d), 4) for n, d in [("snli_test", test), ("mnli_m", mnli_m), ("mnli_mm", mnli_mm)]}
print("fraction of tokens covered by LR's unigram vocabulary:", cov)
json.dump({"vocab_coverage": cov}, open(OUT / "lr_vocab_coverage.json", "w"), indent=2)

fraction of tokens covered by LR's unigram vocabulary: {'snli_test': 0.9872, 'mnli_m': 0.8049, 'mnli_mm': 0.8151}


## Summary

In [8]:
rows = []
for model in ["lr", "lr_hyponly"]:
    for es in ["snli_val", "snli_test", "mnli_m", "mnli_mm"]:
        r = json.load(open(OUT / f"results_{model}_{es}.json"))["metrics"]
        rows.append({"model": model, "eval_set": es, "acc": round(r["accuracy"], 4), "macro_F1": round(r["f1_macro"], 4)})
print(pd.DataFrame(rows).pivot(index="model", columns="eval_set", values="acc")[["snli_val", "snli_test", "mnli_m", "mnli_mm"]])
print("\nFiles in", OUT, ":", sorted(os.listdir(OUT)))
print("Next: Save Version -> Save & Run All; keep this notebook's output (add to the results dataset).")

eval_set    snli_val  snli_test  mnli_m  mnli_mm
model                                           
lr            0.7468     0.7456  0.4918   0.4909
lr_hyponly    0.6451     0.6434  0.4082   0.4155

Files in /kaggle/working/results/lr : ['hyponly_train_oof.csv', 'lr_C_grid.csv', 'lr_feature_ablation.csv', 'lr_model.joblib', 'lr_vocab_coverage.json', 'preds_lr_hyponly_mnli_m.csv', 'preds_lr_hyponly_mnli_mm.csv', 'preds_lr_hyponly_snli_test.csv', 'preds_lr_hyponly_snli_val.csv', 'preds_lr_mnli_m.csv', 'preds_lr_mnli_mm.csv', 'preds_lr_snli_test.csv', 'preds_lr_snli_val.csv', 'results_lr_hyponly_mnli_m.json', 'results_lr_hyponly_mnli_mm.json', 'results_lr_hyponly_snli_test.json', 'results_lr_hyponly_snli_val.json', 'results_lr_mnli_m.json', 'results_lr_mnli_mm.json', 'results_lr_snli_test.json', 'results_lr_snli_val.json']
Next: Save Version -> Save & Run All; keep this notebook's output (add to the results dataset).
